# 05: Full-sequence check of the 27-update candidate

This continues Tiny ImageNet sequence 1 from the saved E08 checkpoint after `L3 L0`. It uses 27 updates for each forget request, forgetting LR `1e-5`, and gamma `5e-6`. It keeps the source checkpoint unchanged, saves after every request, and preserves a separate checkpoint immediately after `U3`.

This is an exploratory reproduction check. The source model learned from all training images, so it is not a valid checkpoint for the reserved-image relearning study. One sequence and one seed cannot establish the full paper result. Use a fresh Colab GPU runtime.


In [ ]:
from urllib.request import urlretrieve
import runpy

urlretrieve(
    "https://raw.githubusercontent.com/sumitasthana/CARK/main/scripts/colab_setup.py",
    "/content/uncle_colab_setup.py",
)
_ = runpy.run_path("/content/uncle_colab_setup.py", run_name="__main__")


## Check the starting checkpoint and settings

The saved model supplies all learning settings. The only changes are the future request list and forgetting settings. Stop if the checkpoint checks fail.


In [ ]:
from pathlib import Path
from uncle.notebook import prepare_session
from uncle import checkpoint as checkpointing
from uncle.config import Config
from uncle.experiments import make_config

SOURCE = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace/before_forgetting.pt")
DATA = Path("/content/data/tiny-imagenet-200")
OUTPUT = SOURCE.parent / "full_sequence_27step_candidate"

session = prepare_session(
    SOURCE, expected_history=[("learn", "3"), ("learn", "0")],
    expected_accuracies={"3": 26.0, "0": 44.6},
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
saved = checkpointing.load(SOURCE)
assert len(saved["history"]) == 2 and saved["forgotten"] == []
full_requests = make_config(sequence=1).requests
assert tuple(full_requests[:2]) == (("learn", "3"), ("learn", "0"))
candidate = Config(**{
    **saved["config"], "requests": full_requests,
    "gamma": 5e-6, "forgetting_learning_rate": 1e-5,
    "burn_in": 27, "burn_in_decay": 1.0, "burn_in_min": 27,
})
del saved
print("Requests:", len(candidate.requests), "seed:", candidate.seed)
print("Forget updates per request: 27 (no annealing)")
print("New output directory:", OUTPUT)


## Continue the complete sequence

This runs the remaining 28 requests. It may take a long time. Progress, JSON logs, and a resumable checkpoint are saved after each request. If the session ends, start a fresh runtime and run these cells again. The same output directory resumes from the last finished request. The source checkpoint is never overwritten.


In [ ]:
from uncle.experiments import run_experiment

result = run_experiment(
    sequence=1, config=candidate, starting_checkpoint=SOURCE,
    root=DATA, download=True, output=OUTPUT,
    checkpoint=True, resume=True, snapshot_after_requests=(2,),
    verbose=True, progress=True,
)


## Read the result

The first forget request is `U3`, at request index 2. The full-sequence numbers measure all requests, including later tasks. The recovery-entry check below is only a planning gate, not a claim of successful unlearning or paper reproduction.


In [ ]:
from uncle.metrics import spill

history = result["history"]
assert len(history) == len(candidate.requests)
first_forget = history[2]
assert (first_forget["action"], first_forget["task"], first_forget["burn_in"]) == ("forget", "3", 27)
drift = abs(first_forget["after"]["0"] - first_forget["before"]["0"])
numbers = result["numbers"]
short_candidate = first_forget["after"]["3"] <= 13.0 and drift < 5.0
recovery_entry = (numbers["retain_accuracy"] is not None
                  and numbers["mean_spill"] is not None
                  and numbers["retain_accuracy"] > 40.0
                  and numbers["mean_spill"] < 3.0)
snapshot = OUTPUT / f"checkpoint_seq1_{candidate.backbone}_seed{candidate.seed}_after_request02_forget3.pt"
assert snapshot.is_file(), snapshot
print("First U3: task 3", first_forget["after"]["3"],
      "task 0", first_forget["after"]["0"],
      "drift", drift, "spill", spill(first_forget))
print("Exploratory <=13% and <5-point short screen:", short_candidate)
print("Full sequence: retain", numbers["retain_accuracy"],
      "forget", numbers["forget_accuracy"],
      "mean spill", numbers["mean_spill"],
      "mean relapse", numbers["mean_relapse"])
print("Recovery-entry gate (>40 retain, <3 mean spill):", recovery_entry)
print("Post-U3 snapshot:", snapshot)
print("Full-sequence summary:", OUTPUT / f"summary_seq1_{candidate.backbone}_seed{candidate.seed}.json")
